# Optimizations and Manipulations

## Preliminary Note on the Data

Since the data is large (1.5 GB once loaded into a dataframe), we'll define one or more functions per exercise to create our charts.

This way, when each function returns, the temporary variables it used are deleted and the memory can be reused.

## Imports

In [ ]:
!pip install python-geohash

import folium
import geohash
import geopy.distance
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import seaborn as sns

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, between 2014 and 2022.

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s dvf

In [ ]:
df = pandas.read_parquet("dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"], inplace=True)

In [ ]:
# For readability, keep the 0 in front of single-digit departments
replacedict = {x: "0" + x for x in df.code_departement.unique() if len(x) == 1}
df["code_departement"] = df["code_departement"].cat.rename_categories(replacedict)

df.info()

## Function Vectorization With numpy

We want a filter telling whether we're within a radius around the city of Nantes

In [ ]:
NANTES_COORDS = 47.218371, -1.553621
folium.Map(location=NANTES_COORDS, zoom_start=14)

To avoid testing all of France, we'll restrict our computation to transactions in Loire-Atlantique

In [ ]:
df_small = df[df.code_departement == "44"]
df_small.shape

We'll use this function from the [`geopy`](https://geopy.readthedocs.io/en/stable/#module-geopy.distance) library for our test

In [ ]:
def is_near(x, y, radius=1):
  return geopy.distance.geodesic(NANTES_COORDS, (x, y)).km < radius

Working on `df_small`, evaluate the difference between the approach with an apply, then with vectorization

### Solution

In [ ]:
radius = 0.5

In [ ]:
is_near_nantes_serie = df_small[["latitude", "longitude"]].apply(
  lambda x: is_near(x["latitude"], x["longitude"], radius), axis=1
)

In [ ]:
vect_funct = numpy.vectorize(lambda x, y: is_near(x, y, radius))
is_near_nantes_vect_funct = vect_funct(df_small["latitude"], df_small["longitude"])

**Why doesn't there seem to be any improvement?**


**Yet we did see improvements in class...**

In [ ]:
geohash_serie = df_small[["latitude", "longitude"]].apply(
  lambda x: geohash.encode(x["latitude"], x["longitude"]), axis=1
)

In [ ]:
vect_funct = numpy.vectorize(lambda x, y: geohash.encode(x, y))
geohash = vect_funct(df_small["latitude"], df_small["longitude"])

In [ ]:
df_small["geohash"] = geohash
df_small = df_small[is_near_nantes_serie]

For function vectorization to bring improvements, the library has to lend itself to it!

## House and Apartment Prices

In this exercise, we'll compute the price per square meter of apartments and houses separately, and display how these prices evolve over the period considered.

- *For apartments, filter the working dataframe to keep only the rows where:*
    - *`nombre_appartements`, `valeur_fonciere` & `surface_reelle_bati_appartements` are strictly positive.*
    - *`nombre_maisons` is zero.*
- *Likewise, for houses, filter the working dataframe to keep only the rows where:*
    - *`nombre_maisons`, `valeur_fonciere` & `surface_reelle_bati_maisons` are strictly positive.*
    - *`nombre_appartements` is zero.*
- *Use this filtered data to compute the price per square meter for each month available in the dataset.*
- *Display the result with the [seaborn.lineplot](https://seaborn.pydata.org/generated/seaborn.lineplot.html) function.*

In [ ]:
def houses_apartments_prices() -> None:
  pass  # Your code here


houses_apartments_prices()

### Solution

In [ ]:
def houses_apartments_prices() -> None:
  # Compute the average price, grouped by date, for houses
  mask_houses = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )
  house_prices = (
    df[mask_houses]
    .groupby(df["date"].dt.to_period("M"))
    .apply(lambda x: sum(x.valeur_fonciere) / sum(x.surface_reelle_bati_maisons))
    .rename("houses")
  )
  # Compute the average price, grouped by date, for apartments
  mask_apartments = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  apt_prices = (
    df[mask_apartments]
    .groupby(df["date"].dt.to_period("M"))
    .apply(lambda x: sum(x.valeur_fonciere) / sum(x.surface_reelle_bati_appartements))
    .rename("apartments")
  )
  # Concatenate the results for display in seaborn
  data = pandas.concat([house_prices, apt_prices], axis=1)
  # The date becomes a column, a new index is created
  data.reset_index(inplace=True)
  # Both series having the same unit, we can reshape our dataframe
  # with a categorical column with two values, "houses" and "apartments",
  # and a column containing the average price
  data = data.melt(id_vars=["date"], value_vars=["houses", "apartments"])
  # seaborn doesn't handle numpy.datetime64 objects
  data.date = data.date.astype("str")

  # Display
  fig, ax = plt.subplots()
  sns.lineplot(data, x="date", y="value", hue="variable", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Average value")
  ax.set_title("Price per square meter")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(
    matplotlib.ticker.EngFormatter(unit="€/m²", places=1)
  )
  fig.show()


houses_apartments_prices()

## Correlations Between Variables

We'll now study the linear correlations with the property value, and more generally between variables.

- *Compute the linear correlations of all variables with `valeur_fonciere` with the [`pandas.DataFrame.corrwith`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html) function.*
- *Sort the variables from the most correlated (or anti-correlated) to the least correlated with `valeur_fonciere`. You can use [`pandas.Series.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.sort_values.html) and its `key` argument, combined with the [`abs`](https://docs.python.org/3/library/functions.html#abs) function.*
- *Compute the correlation matrix of these variables with the [`pandas.DataFrame.corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) function.*
- *Display this matrix with the [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) function.*

In [ ]:
def corrs() -> None:
  pass  # Your code here


corrs()

### Solution

In [ ]:
def corrs() -> None:
  corrs_to_price = (
    df.select_dtypes(include=["number"])
    .corrwith(df.valeur_fonciere)
    .sort_values(ascending=False, key=abs)
  )
  corrs = df.loc[:, corrs_to_price.index].corr()

  fig, ax = plt.subplots(figsize=(15, 10))
  sns.heatmap(corrs, vmin=-1, vmax=1, cmap="vlag", ax=ax)
  ax.set_title("Correlations between variables sorted by correlation to the price")
  fig.show()


corrs()